In [1]:
## config
import duckdb
import pandas as pd
import matplotlib.pyplot as plt


con = duckdb.connect("healthcare.duckdb")

con.execute("PRAGMA threads=8")
con.execute("SET memory_limit='6GB'")

import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

fm.fontManager.addfont(font_path)

font_name = fm.FontProperties(fname=font_path).get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print(font_name)

Noto Sans CJK JP


In [2]:
display(con.execute("""SELECT COUNT(*)
FROM component_atc4_1to1;""").df())
display(con.execute("""SELECT COUNT(*)
FROM component_region
WHERE year = 2022;""").df())
display(con.execute("""SELECT COUNT(*)
FROM market_region_matrix
WHERE matrix_position = 'Validation Candidate';""").df())

,count_star()
0,4411


,count_star()
0,455520


: 

In [2]:
sqlList = []

In [3]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_base AS

SELECT
    CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,
    CAST(diagYm AS VARCHAR) AS diagYm,

    atcStep4Cd,
    atcStep4CdNm,

    st3SickSym,
    st3SickSymNm,

    insupTpCd,

    CAST(totUseQty AS DOUBLE) AS use_qty,
    CAST(msupUseAmt AS DOUBLE) AS use_amt

FROM atc4_sick

WHERE insupTpCd IN ('4','5','7')
  AND atcStep4Cd IS NOT NULL;
""")

In [4]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_yearly AS

SELECT
    year,
    atcStep4Cd,
    MAX(atcStep4CdNm) AS atc4_name,

    SUM(use_amt) AS use_amt,
    SUM(use_qty) AS use_qty

FROM atc4_base

GROUP BY
    year,
    atcStep4Cd;
""")

# SELECT *
# FROM atc4_yearly
# WHERE year = (SELECT MAX(year) FROM atc4_yearly)
# ORDER BY use_amt DESC;

In [5]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_growth AS

WITH p AS (

    SELECT
        atcStep4Cd,
        MAX(atc4_name) AS atc4_name,

        SUM(use_amt) FILTER (WHERE year=2020) AS amt_2020,
        SUM(use_amt) FILTER (WHERE year=2021) AS amt_2021,
        SUM(use_amt) FILTER (WHERE year=2022) AS amt_2022,

        SUM(use_qty) FILTER (WHERE year=2020) AS qty_2020,
        SUM(use_qty) FILTER (WHERE year=2021) AS qty_2021,
        SUM(use_qty) FILTER (WHERE year=2022) AS qty_2022

    FROM atc4_yearly
    GROUP BY atcStep4Cd
)

SELECT
    *,

    amt_2022 - amt_2020 AS growth_amt,

    (amt_2022 - amt_2020)
        / NULLIF(amt_2020,0) AS cumulative_growth_rate,

    CASE
        WHEN amt_2020 > 0
         AND amt_2022 > 0
        THEN POWER(amt_2022 / amt_2020, 1.0/2) - 1
    END AS cagr,

    qty_2022 - qty_2020 AS growth_qty,

    (qty_2022 - qty_2020)
        / NULLIF(qty_2020,0) AS cumulative_qty_growth

FROM p;
""")

In [6]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_persistence AS

SELECT
    g.*,

    CASE
        WHEN amt_2021 > amt_2020 THEN 1
        ELSE 0
    END AS growth_20_21,

    CASE
        WHEN amt_2022 > amt_2021 THEN 1
        ELSE 0
    END AS growth_21_22,

    CASE
        WHEN amt_2021 > amt_2020
         AND amt_2022 > amt_2021
        THEN 1
        ELSE 0
    END AS consistent_growth

FROM atc4_growth g;
""")

In [7]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_market_position AS

WITH r AS (

    SELECT
        *,

        RANK() OVER (
            ORDER BY amt_2022 DESC
        ) AS largest_rank,

        RANK() OVER (
            ORDER BY growth_amt DESC
        ) AS rapid_rank,

        RANK() OVER (
            ORDER BY
                consistent_growth DESC,
                cumulative_growth_rate DESC
        ) AS consistent_rank

    FROM atc4_persistence
)

SELECT
    *,

    CASE

        WHEN largest_rank <= 20
         AND rapid_rank <= 20
         AND consistent_rank <= 20
            THEN '규모·성장·지속성 상위'

        WHEN largest_rank <= 20
         AND rapid_rank BETWEEN 21 AND 39
         AND cumulative_growth_rate > 0
            THEN '대형·성장 중위'

        WHEN largest_rank BETWEEN 21 AND 39
         AND consistent_rank <= 20
         AND cumulative_growth_rate > 0
            THEN '중대형·지속성장 상위'

        WHEN largest_rank > 39
         AND rapid_rank <= 20
         AND cumulative_growth_rate > 0
            THEN '소형·고성장'

        WHEN cumulative_growth_rate < 0
            THEN '역성장'

        ELSE '기타'

    END AS market_layer

FROM r;
""")

In [8]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_disease_yearly AS

SELECT
    CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,

    atcStep4Cd,
    MAX(atcStep4CdNm) AS atc4_name,

    st3SickSym,
    MAX(st3SickSymNm) AS disease_name,

    SUM(CAST(msupUseAmt AS DOUBLE)) AS use_amt,
    SUM(CAST(totUseQty AS DOUBLE)) AS use_qty

FROM atc4_sick

WHERE insupTpCd IN ('4','5','7')
  AND atcStep4Cd IS NOT NULL
  AND st3SickSym IS NOT NULL

GROUP BY
    1,2,4;
""")

In [9]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_disease_share AS

SELECT
    *,

    use_amt
    /
    NULLIF(
        SUM(use_amt) OVER (
            PARTITION BY year, atcStep4Cd
        ),
        0
    ) AS disease_share

FROM atc4_disease_yearly;
""")

In [10]:
sqlList.append("""
CREATE OR REPLACE VIEW atc4_disease_growth AS

SELECT
    atcStep4Cd,
    atc4_name,
    st3SickSym,
    disease_name,

    SUM(use_amt) FILTER (WHERE year=2020) AS amt_2020,
    SUM(use_amt) FILTER (WHERE year=2022) AS amt_2022,

    SUM(use_amt) FILTER (WHERE year=2022)
    -
    SUM(use_amt) FILTER (WHERE year=2020)
        AS growth_amt,

    (
        SUM(use_amt) FILTER (WHERE year=2022)
        -
        SUM(use_amt) FILTER (WHERE year=2020)
    )
    /
    NULLIF(
        SUM(use_amt) FILTER (WHERE year=2020),
        0
    ) AS growth_rate

FROM atc4_disease_yearly

GROUP BY
    atcStep4Cd,
    atc4_name,
    st3SickSym,
    disease_name;
""")
sqlList.append("""
CREATE OR REPLACE VIEW atc4_growth_breadth AS

SELECT
    atcStep4Cd,

    COUNT(*) AS disease_count,

    COUNT(*) FILTER (
        WHERE growth_amt > 0
    ) AS growing_disease_count,

    COUNT(*) FILTER (
        WHERE growth_amt > 0
    ) * 1.0
    /
    NULLIF(COUNT(*),0) AS growth_breadth

FROM atc4_disease_growth

GROUP BY atcStep4Cd;
""")

In [11]:
sqlList.append("""
CREATE OR REPLACE VIEW atc4_top3_growth AS

WITH r AS (

    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY atcStep4Cd
            ORDER BY growth_amt DESC
        ) AS rn

    FROM atc4_disease_growth
),

x AS (

    SELECT
        atcStep4Cd,

        SUM(
            CASE
                WHEN rn <= 3
                 AND growth_amt > 0
                THEN growth_amt
                ELSE 0
            END
        ) AS top3_growth_amt,

        SUM(
            CASE
                WHEN growth_amt > 0
                THEN growth_amt
                ELSE 0
            END
        ) AS total_positive_growth

    FROM r

    GROUP BY atcStep4Cd
)

SELECT
    *,

    top3_growth_amt
    /
    NULLIF(total_positive_growth,0)
        AS top3_growth_contribution

FROM x;
""")
sqlList.append("""
CREATE OR REPLACE VIEW atc4_disease_hhi AS

SELECT
    atcStep4Cd,

    SUM(
        POWER(disease_share,2)
    ) AS disease_hhi

FROM atc4_disease_share

WHERE year = 2022

GROUP BY atcStep4Cd;
""")

In [12]:
sqlList.append("""
CREATE OR REPLACE VIEW atc4_disease_hhi AS

SELECT
    atcStep4Cd,

    SUM(
        POWER(disease_share,2)
    ) AS disease_hhi

FROM atc4_disease_share

WHERE year = 2022

GROUP BY atcStep4Cd;
""")
sqlList.append("""
CREATE OR REPLACE VIEW interest_atc4 AS

SELECT
    m.*,

    b.disease_count,
    b.growing_disease_count,
    b.growth_breadth,

    t.top3_growth_contribution,

    h.disease_hhi

FROM atc4_market_position m

LEFT JOIN atc4_growth_breadth b
    ON m.atcStep4Cd = b.atcStep4Cd

LEFT JOIN atc4_top3_growth t
    ON m.atcStep4Cd = t.atcStep4Cd

LEFT JOIN atc4_disease_hhi h
    ON m.atcStep4Cd = h.atcStep4Cd

WHERE
       m.market_layer <> '역성장'
    OR m.consistent_growth = 1;
""")

In [13]:
sqlList.append("""CREATE OR REPLACE VIEW interest_atc4 AS

SELECT
    m.*,

    b.disease_count,
    b.growing_disease_count,
    b.growth_breadth,

    t.top3_growth_contribution,

    h.disease_hhi

FROM atc4_market_position m

LEFT JOIN atc4_growth_breadth b
    ON m.atcStep4Cd = b.atcStep4Cd

LEFT JOIN atc4_top3_growth t
    ON m.atcStep4Cd = t.atcStep4Cd

LEFT JOIN atc4_disease_hhi h
    ON m.atcStep4Cd = h.atcStep4Cd

WHERE
       m.market_layer <> '역성장'
    OR m.consistent_growth = 1;
""")
sqlList.append("""CREATE OR REPLACE VIEW atc4_region_base AS

SELECT
    CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,

    CAST(diagYm AS VARCHAR) AS diagYm,

    atcStep4Cd,

    regionStep1Cd,
    regionStep1CdNm,

    regionStep2Cd,
    regionStep2CdNm,

    insupTpCd,

    CAST(totUseQty AS DOUBLE) AS use_qty,
    CAST(msupUseAmt AS DOUBLE) AS use_amt

FROM atc4_region

WHERE insupTpCd IN ('4','5','7')
  AND atcStep4Cd IS NOT NULL;
""")

In [14]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_region_yearly AS

SELECT
    year,

    atcStep4Cd,

    regionStep1Cd,
    MAX(regionStep1CdNm) AS region_name,

    SUM(use_amt) AS use_amt,
    SUM(use_qty) AS use_qty

FROM atc4_region_base

GROUP BY
    year,
    atcStep4Cd,
    regionStep1Cd;
""")
sqlList.append("""CREATE OR REPLACE VIEW atc4_region_growth AS

WITH p AS (

    SELECT
        atcStep4Cd,

        regionStep1Cd,
        MAX(region_name) AS region_name,

        SUM(use_amt) FILTER (WHERE year=2020) AS amt_2020,
        SUM(use_amt) FILTER (WHERE year=2021) AS amt_2021,
        SUM(use_amt) FILTER (WHERE year=2022) AS amt_2022

    FROM atc4_region_yearly

    GROUP BY
        atcStep4Cd,
        regionStep1Cd
)

SELECT
    *,

    amt_2022 - amt_2020 AS growth_amt,

    (amt_2022 - amt_2020)
    /
    NULLIF(amt_2020,0)
        AS cumulative_growth_rate,

    CASE
        WHEN amt_2020 > 0
         AND amt_2022 > 0
        THEN POWER(
            amt_2022 / amt_2020,
            1.0/2
        ) - 1
    END AS regional_cagr

FROM p;
""")

In [15]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_region_gap AS

SELECT
    r.*,

    n.cagr AS national_cagr,

    r.regional_cagr - n.cagr AS growth_gap

FROM atc4_region_growth r

LEFT JOIN atc4_growth n
    ON r.atcStep4Cd = n.atcStep4Cd;
""")
sqlList.append("""CREATE OR REPLACE VIEW atc4_regional_dispersion AS

WITH x AS (

    SELECT
        atcStep4Cd,

        regionStep1Cd,

        amt_2022,

        ROW_NUMBER() OVER (
            PARTITION BY atcStep4Cd
            ORDER BY amt_2022
        ) AS rn,

        COUNT(*) OVER (
            PARTITION BY atcStep4Cd
        ) AS n,

        SUM(amt_2022) OVER (
            PARTITION BY atcStep4Cd
        ) AS total_amt

    FROM atc4_region_growth

    WHERE amt_2022 >= 0
),

g AS (

    SELECT
        atcStep4Cd,

        AVG(amt_2022) AS mean_amt,
        STDDEV_SAMP(amt_2022) AS sd_amt,

        SUM(
            rn * amt_2022
        ) AS rank_weighted_sum,

        MAX(n) AS n,
        MAX(total_amt) AS total_amt

    FROM x

    GROUP BY atcStep4Cd
)

SELECT
    *,

    sd_amt / NULLIF(mean_amt,0) AS regional_cv,

    CASE
        WHEN n > 0
         AND total_amt > 0
        THEN
            (
                2.0 * rank_weighted_sum
                / (n * total_amt)
            )
            -
            (n + 1.0) / n
    END AS regional_gini

FROM g;
""")
sqlList.append("""CREATE OR REPLACE VIEW region_context AS

SELECT
    CAST(diagYm AS INTEGER) AS year,

    sidoNm AS region_name,

    TRY_CAST(
        REPLACE(population, ',', '')
        AS DOUBLE
    ) AS population,

    TRY_CAST(
        REPLACE(pharm, ',', '')
        AS DOUBLE
    ) AS pharmacy_count,

    CAST(advGenHosp AS DOUBLE) AS advanced_general_hospital_count,
    CAST(genHosp AS DOUBLE) AS general_hospital_count,
    CAST(hosp AS DOUBLE) AS hospital_count,
    CAST(longHosp AS DOUBLE) AS long_term_hospital_count,

    TRY_CAST(
        REPLACE(clinic, ',', '')
        AS DOUBLE
    ) AS clinic_count,

    TRY_CAST(
        REPLACE(dentalClinic, ',', '')
        AS DOUBLE
    ) AS dental_clinic_count,

    TRY_CAST(
        REPLACE(orientalClinic, ',', '')
        AS DOUBLE
    ) AS oriental_clinic_count

FROM region_medi_facil;
""")

In [16]:
sqlList.append("""CREATE OR REPLACE VIEW atc4_region_institution_base AS

SELECT
    r.atcStep4Cd,
    r.regionStep1Cd,
    r.region_name,

    r.amt_2022,
    r.regional_cagr,
    r.national_cagr,
    r.growth_gap,

    c.population,
    c.pharmacy_count,

    r.amt_2022
    / NULLIF(c.population, 0)
        AS use_amt_per_capita,

    r.amt_2022
    / NULLIF(c.pharmacy_count, 0)
        AS use_amt_per_pharmacy

FROM atc4_region_gap r

LEFT JOIN region_context c
    ON r.region_name = c.region_name
   AND c.year = 2022;
""")
sqlList.append("""CREATE OR REPLACE VIEW atc4_pharmacy_regression AS

SELECT
    atcStep4Cd,

    regr_count(
        LN(1 + amt_2022),
        LN(1 + pharmacy_count)
    ) AS n_obs,

    regr_slope(
        LN(1 + amt_2022),
        LN(1 + pharmacy_count)
    ) AS beta,

    regr_intercept(
        LN(1 + amt_2022),
        LN(1 + pharmacy_count)
    ) AS intercept,

    regr_r2(
        LN(1 + amt_2022),
        LN(1 + pharmacy_count)
    ) AS r2

FROM atc4_region_institution_base

WHERE amt_2022 >= 0
  AND pharmacy_count > 0

GROUP BY atcStep4Cd;
""")
sqlList.append("""CREATE OR REPLACE VIEW atc4_institution_residual AS

SELECT
    r.*,

    p.beta,
    p.intercept,
    p.r2,

    LN(1 + r.amt_2022)
    -
    (
        p.intercept
        +
        p.beta * LN(1 + r.pharmacy_count)
    ) AS institution_residual

FROM atc4_region_institution_base r

LEFT JOIN atc4_pharmacy_regression p
    ON r.atcStep4Cd = p.atcStep4Cd;
""")

In [17]:
sqlList.append("""CREATE OR REPLACE VIEW market_region AS

SELECT
    r.atcStep4Cd,
    r.regionStep1Cd,
    r.region_name,

    /* National market */
    m.atc4_name,
    m.amt_2022 AS national_market_amt,
    m.cagr AS national_cagr,
    m.consistent_growth,
    m.market_layer,

    /* Disease */
    b.disease_count,
    b.growing_disease_count,
    b.growth_breadth,

    t.top3_growth_contribution,

    h.disease_hhi,

    /* Regional */
    r.amt_2022 AS regional_market_amt,
    r.regional_cagr,
    r.growth_gap,

    d.regional_cv,
    d.regional_gini,

    /* Structural context */
    i.population,
    i.pharmacy_count,
    i.use_amt_per_capita,
    i.use_amt_per_pharmacy,
    i.institution_residual,
    i.r2 AS institution_model_r2

FROM atc4_region_gap r

JOIN atc4_market_position m
    ON r.atcStep4Cd = m.atcStep4Cd

LEFT JOIN atc4_growth_breadth b
    ON r.atcStep4Cd = b.atcStep4Cd

LEFT JOIN atc4_top3_growth t
    ON r.atcStep4Cd = t.atcStep4Cd

LEFT JOIN atc4_disease_hhi h
    ON r.atcStep4Cd = h.atcStep4Cd

LEFT JOIN atc4_regional_dispersion d
    ON r.atcStep4Cd = d.atcStep4Cd

LEFT JOIN atc4_institution_residual i
    ON r.atcStep4Cd = i.atcStep4Cd
   AND r.regionStep1Cd = i.regionStep1Cd;
""")
sqlList.append("""CREATE OR REPLACE VIEW market_region_matrix AS

WITH q AS (

    SELECT
        *,

        NTILE(4) OVER (
            ORDER BY national_market_amt
        ) AS national_size_q,

        NTILE(4) OVER (
            ORDER BY national_cagr
        ) AS national_growth_q,

        NTILE(4) OVER (
            ORDER BY regional_market_amt
        ) AS regional_size_q,

        NTILE(4) OVER (
            ORDER BY growth_gap
        ) AS growth_gap_q

    FROM market_region
)

SELECT
    *,

    CASE
        WHEN national_size_q >= 3
         AND national_growth_q >= 3
        THEN 'High'
        ELSE 'Lower'
    END AS market_attractiveness,

    CASE
        WHEN regional_size_q >= 3
         AND growth_gap_q >= 3
        THEN 'High'
        ELSE 'Lower'
    END AS regional_opportunity,

    CASE

        WHEN national_size_q >= 3
         AND national_growth_q >= 3
         AND regional_size_q >= 3
         AND growth_gap_q >= 3
            THEN 'Validation Candidate'

        WHEN national_size_q >= 3
         AND national_growth_q >= 3
         AND NOT (
             regional_size_q >= 3
             AND growth_gap_q >= 3
         )
            THEN 'Core Market'

        WHEN NOT (
             national_size_q >= 3
             AND national_growth_q >= 3
        )
        AND regional_size_q >= 3
        AND growth_gap_q >= 3
            THEN 'Regional Niche'

        ELSE 'Monitor'

    END AS matrix_position

FROM q;
""")
sqlList.append("""CREATE OR REPLACE VIEW component_atc4_cardinality AS

SELECT
    MainCmpCd AS gnlNmCd,

    COUNT(DISTINCT atc4cd) AS atc4_count,

    LIST(
        DISTINCT atc4cd
        ORDER BY atc4cd
    ) AS atc4_list

FROM prod_atc_map

WHERE MainCmpCd IS NOT NULL
  AND atc4cd IS NOT NULL

GROUP BY MainCmpCd;
""")

# SELECT *
# FROM component_atc4_cardinality
# WHERE atc4_count > 1
# ORDER BY atc4_count DESC;

In [18]:
sqlList.append("""CREATE OR REPLACE VIEW component_atc4_1to1 AS

SELECT
    MainCmpCd AS gnlNmCd,
    MIN(atc4cd) AS atcStep4Cd

FROM prod_atc_map

WHERE MainCmpCd IS NOT NULL
  AND atc4cd IS NOT NULL

GROUP BY MainCmpCd

HAVING COUNT(DISTINCT atc4cd) = 1;
""")
sqlList.append("""CREATE OR REPLACE VIEW component_market AS

SELECT
    m.atcStep4Cd,

    c.gnlNmCd,
    MAX(c.gnlNmCdNm) AS component_name,

    SUM(c.msupUseAmt) AS use_amt,
    SUM(c.totUseQty) AS use_qty

FROM cmpn_sick c

JOIN component_atc4_1to1 m
    ON c.gnlNmCd = m.gnlNmCd

WHERE c.insupTpCd IN ('4','5','7')

GROUP BY
    m.atcStep4Cd,
    c.gnlNmCd;
""")
sqlList.append("""
""")

In [19]:
sqlList.append("""CREATE OR REPLACE VIEW component_market_ranked AS

SELECT
    *,

    RANK() OVER (
        PARTITION BY atcStep4Cd
        ORDER BY use_amt DESC
    ) AS component_rank

FROM component_market;
""")
sqlList.append("""CREATE OR REPLACE VIEW component_disease AS

SELECT
    CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,

    gnlNmCd,
    gnlNmCdNm AS component_name,

    st3SickSym,
    st3SickSymNm AS disease_name,

    SUM(msupUseAmt) AS use_amt,
    SUM(totUseQty) AS use_qty

FROM cmpn_sick

WHERE insupTpCd IN ('4','5','7')

GROUP BY
    1,
    2,
    3,
    4,
    5;
""")
sqlList.append("""
CREATE OR REPLACE VIEW component_region_2022 AS
SELECT
    gnlNmCd,
    gnlNmCdNm AS component_name,
    regionStep1Cd,
    regionStep1CdNm AS region_name,
    regionStep2Cd,
    regionStep2CdNm AS district_name,
    SUM(totUseQty) AS use_qty,
    SUM(msupUseAmt) AS use_amt
FROM cmpn_region
WHERE LEFT(CAST(diagYm AS VARCHAR), 4) = '2022'
  AND insupTpCd IN ('4','5','7')
GROUP BY
    gnlNmCd,
    gnlNmCdNm,
    regionStep1Cd,
    regionStep1CdNm,
    regionStep2Cd,
    regionStep2CdNm;
""")

# SELECT COUNT(*)
# FROM component_region_2022;

In [20]:
sqlList.append("""CREATE OR REPLACE VIEW candidate_component_region_base AS
SELECT
    m.atcStep4Cd,
    m.atc4_name,
    m.regionStep1Cd,
    m.region_name,
    cm.gnlNmCd,
    cr.component_name,
    cr.regionStep2Cd,
    cr.district_name,
    cr.use_amt,
    cr.use_qty
FROM market_region_matrix AS m

JOIN component_atc4_1to1 AS cm
    ON m.atcStep4Cd = cm.atcStep4Cd

JOIN component_region_2022 AS cr
    ON cm.gnlNmCd = cr.gnlNmCd
   AND m.regionStep1Cd = cr.regionStep1Cd

WHERE m.matrix_position = 'Validation Candidate';
""")
sqlList.append("""
CREATE OR REPLACE VIEW candidate_component_region AS
SELECT
    atcStep4Cd,
    atc4_name,
    regionStep1Cd,
    region_name,
    gnlNmCd,
    component_name,
    regionStep2Cd,
    district_name,
    SUM(use_amt) AS use_amt,
    SUM(use_qty) AS use_qty
FROM candidate_component_region_base
GROUP BY
    atcStep4Cd,
    atc4_name,
    regionStep1Cd,
    region_name,
    gnlNmCd,
    component_name,
    regionStep2Cd,
    district_name;
""")
sqlList.append("""CREATE OR REPLACE VIEW final_validation AS

SELECT
    matrix_position,

    /* Market */
    atcStep4Cd,
    atc4_name,

    national_market_amt,
    national_cagr,
    consistent_growth,
    market_layer,

    /* Demand */
    disease_count,
    growing_disease_count,
    growth_breadth,
    top3_growth_contribution,
    disease_hhi,

    /* Region */
    regionStep1Cd,
    region_name,

    regional_market_amt,
    regional_cagr,
    growth_gap,

    regional_cv,
    regional_gini,

    /* Structure */
    population,
    pharmacy_count,
    use_amt_per_capita,
    use_amt_per_pharmacy,

    institution_residual,
    institution_model_r2,

    /* Matrix */
    market_attractiveness,
    regional_opportunity

FROM market_region_matrix;
""")
# SELECT *
# FROM final_validation

# WHERE matrix_position = 'Validation Candidate'

# ORDER BY
#     national_market_amt DESC,
#     regional_market_amt DESC;

In [ ]:
for isql in sqlList:
    print(isql.splitlines()[0])
    con.execute(isql)

CREATE OR REPLACE VIEW atc4_base AS
CREATE OR REPLACE VIEW atc4_yearly AS
CREATE OR REPLACE VIEW atc4_growth AS
CREATE OR REPLACE VIEW atc4_persistence AS
CREATE OR REPLACE VIEW atc4_market_position AS
CREATE OR REPLACE VIEW atc4_disease_yearly AS
CREATE OR REPLACE VIEW atc4_disease_share AS






CREATE OR REPLACE VIEW interest_atc4 AS
CREATE OR REPLACE VIEW atc4_region_base AS
CREATE OR REPLACE VIEW atc4_region_yearly AS
CREATE OR REPLACE VIEW atc4_region_growth AS
CREATE OR REPLACE VIEW atc4_region_gap AS
CREATE OR REPLACE VIEW atc4_regional_dispersion AS
CREATE OR REPLACE VIEW region_context AS
CREATE OR REPLACE VIEW atc4_region_institution_base AS
CREATE OR REPLACE VIEW atc4_pharmacy_regression AS
CREATE OR REPLACE VIEW atc4_institution_residual AS
CREATE OR REPLACE VIEW market_region AS
CREATE OR REPLACE VIEW market_region_matrix AS
CREATE OR REPLACE VIEW component_atc4_cardinality AS
CREATE OR REPLACE VIEW component_atc4_1to1 AS
CREATE OR REPLACE VIEW component_market AS

CREATE

: 

In [ ]:
con.execute("""
SELECT
    f.atcStep4Cd,
    f.atc4_name,

    f.regionStep1Cd,
    f.region_name,

    f.national_market_amt,
    f.national_cagr,

    f.regional_market_amt,
    f.regional_cagr,
    f.growth_gap,

    f.growth_breadth,
    f.disease_hhi,

    c.gnlNmCd,
    c.component_name,
    c.use_amt AS component_use_amt,
    c.use_qty AS component_use_qty,
    c.component_rank

FROM final_validation f

JOIN component_market_ranked c
    ON f.atcStep4Cd = c.atcStep4Cd

WHERE f.matrix_position = 'Validation Candidate'

ORDER BY
    f.national_market_amt DESC,
    f.regional_market_amt DESC,
    c.use_amt DESC;
""")